# Skrub head ablation

Follow-up to `../representation_bakeoff/`. The bake-off fixed the **head** (one MLP) and
varied the **representation**; skrub `TableVectorizer` won (AUC 0.5982 ± 0.0014).
This ablation does the opposite: the **skrub representation is frozen** and only the
**classifier head** varies.

Held identical across every row (same as the bake-off):
- **Sample:** the bake-off's fixed 50k `sample.parquet` (all years 2016–2024, stratified by year + label).
- **Split:** temporal by `FL_DAY` — train ≤ 9 / val 10–12 / test > 12. Never shuffled across days.
- **Target:** `|ARR_DELAY| > 15 min`.
- **Representation:** skrub `TableVectorizer(cardinality_threshold=40)`, fit on train rows only, then `StandardScaler` fit on train.
- **Metric:** test AUC (primary) + F1-macro + accuracy, mean ± std over seeds `[42, 43, 44]`. Torch heads early-stop on **val AUC**.

The `mlp` head is the bake-off head unchanged, so its row should reproduce ≈ 0.598 — that is the
reference every other head is compared against.

**To add or change a head:** add an entry to `HEADS` in the config cell (and, for a new architecture,
a class + a line in `build_torch_head`). Results are appended to `results.csv`; heads already
there are skipped unless `FORCE_RERUN = True`.

In [ ]:
# Uncomment on a fresh machine (Colab / Kaggle / remote box).
# !pip install -q "skrub>=0.10.0" pyarrow pyyaml scikit-learn pandas matplotlib torch

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

## Config

Edit the paths for the machine you run on. Everything else mirrors
`../representation_bakeoff/config.yaml` so the rows stay comparable with the bake-off.

In [ ]:
# ---- Paths ------------------------------------------------------------------
# The bake-off's fixed sample (written by representation_bakeoff/prep/build_sample.py).
# Copy it next to this notebook when running elsewhere.
SAMPLE_PATH = Path("../representation_bakeoff/artifacts/sample.parquet")
# Fallback only if SAMPLE_PATH is missing: folder holding Flight_tab_YYYY.csv +
# data_info_YYYY.yaml for every year below; the sample is then rebuilt identically.
TAB_DIR = None                      # e.g. Path("/content/Tab")
OUT_DIR = Path("outputs")           # results.csv, cached embedding, figures

# ---- Data (identical to the bake-off) ---------------------------------------
DATA = dict(
    years=[2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    sample_total=50000,
    seed=42,
    train_days_max=9,               # FL_DAY <= 9  -> train
    val_days_max=12,                # 9 < FL_DAY <= 12 -> val ; > 12 -> test
    delay_threshold=15.0,           # |ARR_DELAY| > 15 -> label 1
    exclude_cols=["FLIGHTS", "FL_YEAR", "FL_MONTH"],
)
# Schema from Datasets/Flight_tab.py (identical across years). DEP_DELAY / ARR_DELAY are targets.
_CAT_ALL = ["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_YEAR", "FL_MONTH", "FL_DAY", "FL_WEEK",
            "ORIGIN_INDEX", "DEST_INDEX"]
_CONT_ALL = ["CRS_DEP_TIME_MIN", "CRS_ARR_TIME_MIN", "CRS_ELAPSED_TIME", "FLIGHTS",
             "O_TEMP", "O_PRCP", "O_WSPD", "D_TEMP", "D_PRCP", "D_WSPD",
             "O_LATITUDE", "O_LONGITUDE", "D_LATITUDE", "D_LONGITUDE"]
CAT = [c for c in _CAT_ALL if c not in DATA["exclude_cols"]]
CONT = [c for c in _CONT_ALL if c not in DATA["exclude_cols"]]

# ---- Representation (frozen) ------------------------------------------------
SKRUB = dict(cardinality_threshold=40)
REUSE_EMBEDDING = True              # load OUT_DIR/emb_skrub.npy if it matches the sample

# ---- Shared training protocol -----------------------------------------------
TRAIN = dict(
    seeds=[42, 43, 44],
    batch_size=2048,                # per-head override via HEADS[...]["batch_size"]
    max_epochs=50,
    patience=5,                     # early stop on val AUC
    eval_batch_size=4096,           # batched inference (attention heads on 30k test rows)
)

# ---- Heads ------------------------------------------------------------------
# kind="torch": trained with the shared loop below (Adam/AdamW, BCE, early stop on val AUC).
# kind="sklearn": fit on train; val is not used (HGB early-stops on an internal train slice).
HEADS = {
    # Bake-off head, unchanged -> reference row (expect AUC ~0.598).
    "mlp": dict(kind="torch", arch="mlp", hidden=[256, 128], dropout=0.1,
                optimizer="adam", lr=1e-3, weight_decay=0.0),
    # Same MLP family, wider/deeper with BatchNorm + AdamW.
    "mlp_bn_wide": dict(kind="torch", arch="mlp", hidden=[512, 256, 128], dropout=0.2,
                        batchnorm=True, optimizer="adamw", lr=1e-3, weight_decay=1e-4),
    # Residual MLP (Gorishniy et al., 2021).
    "resnet": dict(kind="torch", arch="resnet", d=256, d_hidden=512, n_blocks=3,
                   dropout=0.2, optimizer="adamw", lr=1e-3, weight_decay=1e-5),
    # FT-Transformer: every skrub column becomes a token; head reads the [CLS] token.
    "ft_transformer": dict(kind="torch", arch="ft_transformer", d_token=64, n_layers=3,
                           n_heads=8, ffn_mult=4 / 3, dropout=0.1, optimizer="adamw",
                           lr=1e-4, weight_decay=1e-5, batch_size=512),
    # Non-neural reference points.
    "logreg": dict(kind="sklearn", model="logreg", C=1.0),
    "hist_gbm": dict(kind="sklearn", model="hist_gbm", learning_rate=0.05, max_iter=500,
                     max_leaf_nodes=31, l2_regularization=1.0),
}
HEADS_TO_RUN = list(HEADS)          # or e.g. ["mlp", "resnet"]
FORCE_RERUN = False                 # True -> recompute heads already in results.csv

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(CAT)} categorical + {len(CONT)} continuous features | heads: {HEADS_TO_RUN}")

## Load the fixed sample

Reads `sample.parquet`. If it is missing and `TAB_DIR` is set, the sample is rebuilt with the
bake-off's exact procedure (`prep/build_sample.py`: per-year quota proportional to size,
label-stratified, `random_state=42`) and cached to `OUT_DIR/sample.parquet`.

In [ ]:
def build_sample(tab_dir: Path, data: dict) -> pd.DataFrame:
    import yaml
    from sklearn.model_selection import train_test_split

    counts = {}
    for y in data["years"]:
        with open(tab_dir / f"data_info_{y}.yaml") as f:
            counts[y] = int(yaml.safe_load(f)["data_summary"]["shape"][0])

    # largest-remainder allocation of sample_total, proportional to year size
    total, grand = data["sample_total"], sum(counts.values())
    exact = {y: total * c / grand for y, c in counts.items()}
    quota = {y: int(np.floor(v)) for y, v in exact.items()}
    for y in sorted(exact, key=lambda y: exact[y] - quota[y], reverse=True)[: total - sum(quota.values())]:
        quota[y] += 1

    parts = []
    for y in data["years"]:
        if quota[y] <= 0:
            continue
        d = pd.read_csv(tab_dir / f"Flight_tab_{y}.csv").dropna()
        d["label"] = (d["ARR_DELAY"].abs() > data["delay_threshold"]).astype(int)
        d["FL_YEAR"] = y
        sub, _ = train_test_split(d, train_size=min(quota[y], len(d)),
                                  stratify=d["label"], random_state=data["seed"])
        parts.append(sub)
        print(f"  [{y}] sampled {len(sub)}")
        del d

    s = pd.concat(parts, ignore_index=True)
    s["split"] = np.select([s["FL_DAY"] <= data["train_days_max"],
                            s["FL_DAY"] <= data["val_days_max"]], ["train", "val"], default="test")
    return s


if SAMPLE_PATH.exists():
    df = pd.read_parquet(SAMPLE_PATH)
elif (OUT_DIR / "sample.parquet").exists():
    df = pd.read_parquet(OUT_DIR / "sample.parquet")
elif TAB_DIR is not None:
    df = build_sample(Path(TAB_DIR), DATA)
    df.to_parquet(OUT_DIR / "sample.parquet", index=False)
else:
    raise FileNotFoundError(f"{SAMPLE_PATH} not found and TAB_DIR is None — set one of them.")

missing = [c for c in CAT + CONT + ["label", "split"] if c not in df.columns]
assert not missing, f"sample is missing columns: {missing}"

tr_m = (df["split"] == "train").to_numpy()
va_m = (df["split"] == "val").to_numpy()
te_m = (df["split"] == "test").to_numpy()
y = df["label"].to_numpy().astype(int)
y_tr, y_va, y_te = y[tr_m], y[va_m], y[te_m]

print(f"rows: {len(df)} | train/val/test = {tr_m.sum()}/{va_m.sum()}/{te_m.sum()} | "
      f"delayed frac = {y.mean():.3f}")

## Frozen skrub representation

Identical to `representation_bakeoff/embeddings/embed_skrub.py`: categoricals cast to `str`,
`TableVectorizer` fit on **train rows only**, then transformed for all rows. Computed once and
shared by every head. A `StandardScaler` fit on train is applied on top (the bake-off head did
the same inside each seed — deterministic, so doing it once is equivalent).

In [ ]:
emb_path = OUT_DIR / "emb_skrub.npy"
Z = np.load(emb_path) if (REUSE_EMBEDDING and emb_path.exists()) else None
if Z is not None and Z.shape[0] != len(df):
    print(f"cached embedding has {Z.shape[0]} rows, sample has {len(df)} -> recomputing")
    Z = None

if Z is None:
    from skrub import TableVectorizer

    X = df[CAT + CONT].copy()
    for c in CAT:
        X[c] = X[c].astype(str)
    tv = TableVectorizer(cardinality_threshold=SKRUB["cardinality_threshold"])
    tv.fit(X[tr_m])
    Z = tv.transform(X)
    Z = np.asarray(getattr(Z, "values", Z), dtype=np.float32)
    np.save(emb_path, Z)

scaler = StandardScaler().fit(Z[tr_m])
Z_tr = scaler.transform(Z[tr_m]).astype(np.float32)
Z_va = scaler.transform(Z[va_m]).astype(np.float32)
Z_te = scaler.transform(Z[te_m]).astype(np.float32)
print(f"skrub dim = {Z.shape[1]}  (bake-off: 95)")

## Heads

All torch heads map `(batch, dim) -> (batch,)` logits.

In [ ]:
class MLP(nn.Module):
    # batchnorm=False reproduces the bake-off head (head/mlp_head.py) exactly.
    def __init__(self, in_dim, hidden, dropout, batchnorm=False):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h)]
            if batchnorm:
                layers += [nn.BatchNorm1d(h)]
            layers += [nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class ResBlock(nn.Module):
    def __init__(self, d, d_hidden, dropout):
        super().__init__()
        self.norm = nn.BatchNorm1d(d)
        self.ff = nn.Sequential(nn.Linear(d, d_hidden), nn.ReLU(), nn.Dropout(dropout),
                                nn.Linear(d_hidden, d), nn.Dropout(dropout))

    def forward(self, x):
        return x + self.ff(self.norm(x))


class ResNet(nn.Module):
    def __init__(self, in_dim, d, d_hidden, n_blocks, dropout):
        super().__init__()
        self.inp = nn.Linear(in_dim, d)
        self.blocks = nn.Sequential(*[ResBlock(d, d_hidden, dropout) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, 1))

    def forward(self, x):
        return self.out(self.blocks(self.inp(x))).squeeze(-1)


class FTTransformer(nn.Module):
    # Numerical tokenizer: column j -> x_j * W_j + b_j, then [CLS] + pre-norm Transformer.
    def __init__(self, in_dim, d_token, n_layers, n_heads, ffn_mult, dropout):
        super().__init__()
        self.W = nn.Parameter(torch.randn(in_dim, d_token) * d_token ** -0.5)
        self.b = nn.Parameter(torch.randn(in_dim, d_token) * d_token ** -0.5)
        self.cls = nn.Parameter(torch.randn(1, 1, d_token) * d_token ** -0.5)
        layer = nn.TransformerEncoderLayer(d_token, n_heads, int(d_token * ffn_mult), dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.ReLU(), nn.Linear(d_token, 1))

    def forward(self, x):
        tok = x.unsqueeze(-1) * self.W + self.b
        tok = torch.cat([self.cls.expand(len(x), -1, -1), tok], dim=1)
        return self.head(self.encoder(tok)[:, 0]).squeeze(-1)


def build_torch_head(in_dim, hp):
    if hp["arch"] == "mlp":
        return MLP(in_dim, hp["hidden"], hp["dropout"], hp.get("batchnorm", False))
    if hp["arch"] == "resnet":
        return ResNet(in_dim, hp["d"], hp["d_hidden"], hp["n_blocks"], hp["dropout"])
    if hp["arch"] == "ft_transformer":
        return FTTransformer(in_dim, hp["d_token"], hp["n_layers"], hp["n_heads"],
                             hp["ffn_mult"], hp["dropout"])
    raise ValueError(f"unknown arch {hp['arch']!r}")


def build_sklearn_head(hp, seed):
    if hp["model"] == "logreg":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(C=hp["C"], max_iter=2000, random_state=seed)
    if hp["model"] == "hist_gbm":
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(
            learning_rate=hp["learning_rate"], max_iter=hp["max_iter"],
            max_leaf_nodes=hp["max_leaf_nodes"], l2_regularization=hp["l2_regularization"],
            early_stopping=True, validation_fraction=0.1, scoring="roc_auc", random_state=seed)
    raise ValueError(f"unknown model {hp['model']!r}")

## Training / evaluation

The torch loop is the bake-off's (`head/mlp_head.py`): seed → build model → optimizer →
seeded shuffling loader; early stop on val AUC with `patience`, restore the best state, then
threshold test probabilities at 0.5. The only additions are a per-head optimizer / weight decay /
batch size and batched inference.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict_proba(model, X):
    model.eval()
    bs = TRAIN["eval_batch_size"]
    return np.concatenate([torch.sigmoid(model(X[i:i + bs])).cpu().numpy()
                           for i in range(0, len(X), bs)])


def safe_auc(y_true, p):
    return roc_auc_score(y_true, p) if len(np.unique(y_true)) > 1 else float("nan")


def train_torch(hp, seed):
    set_seed(seed)
    Xtr = torch.as_tensor(Z_tr, device=DEVICE)
    ytr = torch.as_tensor(y_tr, dtype=torch.float32, device=DEVICE)
    Xva = torch.as_tensor(Z_va, device=DEVICE)
    Xte = torch.as_tensor(Z_te, device=DEVICE)

    model = build_torch_head(Xtr.shape[1], hp).to(DEVICE)
    opt_cls = torch.optim.AdamW if hp.get("optimizer", "adam") == "adamw" else torch.optim.Adam
    opt = opt_cls(model.parameters(), lr=hp["lr"], weight_decay=hp.get("weight_decay", 0.0))
    loss_fn = nn.BCEWithLogitsLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr),
        batch_size=hp.get("batch_size", TRAIN["batch_size"]),
        shuffle=True, generator=torch.Generator().manual_seed(seed))

    best_auc, best_state, best_epoch, no_improve = -np.inf, None, 0, 0
    for epoch in range(TRAIN["max_epochs"]):
        model.train()
        for xb, yb in loader:
            if len(xb) < 2 and any(isinstance(m, nn.BatchNorm1d) for m in model.modules()):
                continue  # BatchNorm cannot train on a single-row tail batch
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        auc = safe_auc(y_va, predict_proba(model, Xva))
        auc = 0.5 if np.isnan(auc) else auc
        if auc > best_auc:
            best_auc, best_epoch, no_improve = auc, epoch + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= TRAIN["patience"]:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    n_params = sum(p.numel() for p in model.parameters())
    return predict_proba(model, Xte), dict(val_auc=best_auc, best_epoch=best_epoch, n_params=n_params)


def train_sklearn(hp, seed):
    model = build_sklearn_head(hp, seed).fit(Z_tr, y_tr)
    val_auc = safe_auc(y_va, model.predict_proba(Z_va)[:, 1])
    extra = dict(val_auc=val_auc, best_epoch=getattr(model, "n_iter_", np.nan), n_params=np.nan)
    if isinstance(extra["best_epoch"], np.ndarray):  # LogisticRegression.n_iter_ is an array
        extra["best_epoch"] = int(extra["best_epoch"].max())
    return model.predict_proba(Z_te)[:, 1], extra


def run_head(name, hp):
    rows = []
    for seed in TRAIN["seeds"]:
        t0 = time.time()
        p, extra = (train_torch if hp["kind"] == "torch" else train_sklearn)(hp, seed)
        pred = (p > 0.5).astype(int)
        m = dict(auc=safe_auc(y_te, p), f1=f1_score(y_te, pred, average="macro"),
                 acc=accuracy_score(y_te, pred), time_s=time.time() - t0, **extra)
        rows.append(m)
        print(f"    seed={seed}: AUC={m['auc']:.4f} F1m={m['f1']:.4f} ACC={m['acc']:.4f} "
              f"| val AUC={m['val_auc']:.4f} @ {m['best_epoch']} | {m['time_s']:.0f}s")
    r = pd.DataFrame(rows)
    return {
        "Head": name, "dim": Z_tr.shape[1], "n_params": r["n_params"].iloc[0],
        "auc_mean": r["auc"].mean(), "auc_std": r["auc"].std(ddof=0),
        "f1_mean": r["f1"].mean(), "f1_std": r["f1"].std(ddof=0),
        "acc_mean": r["acc"].mean(), "acc_std": r["acc"].std(ddof=0),
        "val_auc_mean": r["val_auc"].mean(), "time_s_mean": r["time_s"].mean(),
        "params": json.dumps({k: v for k, v in hp.items() if k != "kind"}),
    }

## Run

Results are written to `OUT_DIR/results.csv` after every head, so an interrupted run resumes
where it stopped.

In [ ]:
results_csv = OUT_DIR / "results.csv"
results = pd.read_csv(results_csv) if results_csv.exists() else pd.DataFrame()

for name in HEADS_TO_RUN:
    done = not results.empty and name in results["Head"].values
    if done and not FORCE_RERUN:
        print(f"Skipping {name} — already in results.csv")
        continue
    print(f"{'=' * 60}\n  {name}\n{'=' * 60}")
    try:
        row = run_head(name, HEADS[name])
    except Exception:
        import traceback
        print(f"  !! {name} FAILED:\n{traceback.format_exc()}")
        continue
    print(f"  -> AUC {row['auc_mean']:.4f} +/- {row['auc_std']:.4f} | "
          f"F1m {row['f1_mean']:.4f} | ACC {row['acc_mean']:.4f}")
    if done:
        results = results[results["Head"] != name]
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    results.to_csv(results_csv, index=False)

print(f"\nResults -> {results_csv}")

## Results

`Δ AUC` is relative to the `mlp` row (the bake-off head). `σ` is how many seed-stds the gap is,
using the pooled std of the two rows — a rough guide with only 3 seeds.

In [ ]:
res = pd.read_csv(results_csv).sort_values("auc_mean", ascending=False).reset_index(drop=True)
if "mlp" in res["Head"].values:
    ref = res.set_index("Head").loc["mlp"]
    res["d_auc"] = res["auc_mean"] - ref["auc_mean"]
    pooled = np.sqrt((res["auc_std"] ** 2 + ref["auc_std"] ** 2) / 2).replace(0, np.nan)
    res["sigma"] = res["d_auc"] / pooled

cols = [c for c in ["Head", "n_params", "auc_mean", "auc_std", "d_auc", "sigma",
                    "f1_mean", "acc_mean", "val_auc_mean", "time_s_mean"] if c in res]
res[cols].style.format({
    "n_params": "{:,.0f}", "auc_mean": "{:.4f}", "auc_std": "{:.4f}", "d_auc": "{:+.4f}",
    "sigma": "{:+.1f}", "f1_mean": "{:.4f}", "acc_mean": "{:.4f}", "val_auc_mean": "{:.4f}",
    "time_s_mean": "{:.0f}"}, na_rep="—")

In [ ]:
import matplotlib.pyplot as plt

# Dot + whisker (AUC differences are small; bars from 0 would hide them).
# Colors follow representation_bakeoff/make_figures.py (light theme).
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9"
ACCENT, REF = "#2a78d6", "#898781"

p = res.sort_values("auc_mean")
fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(p) + 1.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
colors = [REF if h == "mlp" else ACCENT for h in p["Head"]]
for i, (a, s, c) in enumerate(zip(p["auc_mean"], p["auc_std"], colors)):
    ax.hlines(i, a - s, a + s, color=c, lw=2, zorder=2)
ax.scatter(p["auc_mean"], range(len(p)), s=64, c=colors, edgecolors=SURFACE,
           linewidths=2, zorder=3)
if "mlp" in p["Head"].values:
    ax.axvline(float(p.loc[p["Head"] == "mlp", "auc_mean"].iloc[0]), color=REF,
               lw=1, ls=(0, (4, 3)), zorder=1)
for i, (a, s) in enumerate(zip(p["auc_mean"], p["auc_std"])):
    ax.text(a + s + 0.0008, i, f"{a:.4f}", va="center", fontsize=9, color=INK2)

ax.set_yticks(range(len(p)), [f"{h}  (reference)" if h == "mlp" else h for h in p["Head"]],
              color=INK)
ax.set_xlabel("Test AUC (mean ± std over seeds)", color=INK2)
ax.set_title("Skrub representation, head varies", loc="left", color=INK, fontsize=12)
ax.grid(axis="x", color=GRID, lw=1)
ax.tick_params(colors=INK2, length=0)
for s in ax.spines.values():
    s.set_visible(False)
lo, hi = (p["auc_mean"] - p["auc_std"]).min(), (p["auc_mean"] + p["auc_std"]).max()
ax.set_xlim(lo - 0.003, hi + 0.006)
fig.tight_layout()

(OUT_DIR / "figures").mkdir(exist_ok=True)
fig.savefig(OUT_DIR / "figures" / "head_ablation.png", dpi=200, facecolor=SURFACE)
plt.show()